In [1]:
import torch
import torch.nn as nn
import math

In [ ]:
class TimeEmbedding(nn.Module):
    # PE(t, 2i) = sin(t / 10000^(2i/d))
    # PE(t, 2i+1) = cos(t / 10000^(2i/d))
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, t):
        # t shape: (batch,)
        device = t.device
        half_dim = self.dim // 2
        i = torch.arange(0, half_dim, device=device) 
        div_term = 10000 ** (2 * i / self.dim)
        
        # t needs to be (batch, 1) to broadcast against div_term (half_dim,)
        t = t.unsqueeze(1).float()
        
        sins = torch.sin(t/div_term)
        coss = torch.cos(t/div_term)
        emb = torch.cat((sins, coss), dim=-1)
        return emb  # shape: (batch, dim)

In [2]:
d = 16
t = 50

In [5]:
num = torch.arange(0, d)

In [10]:
ieven, iodd = num[num % 2 == 0] , num[num % 2 != 0]

In [11]:
torch.sin(t/10000**((2*ieven)/d))

tensor([-2.6237e-01, -9.5892e-01,  4.7943e-01,  4.9979e-02,  5.0000e-03,
         5.0000e-04,  5.0000e-05,  5.0000e-06])

In [12]:
torch.cos(t/10000**((2*iodd)/d))

tensor([-0.9947, -0.0103,  0.9875,  0.9999,  1.0000,  1.0000,  1.0000,  1.0000])

In [15]:
embeddings =  TimeEmbedding(dim=d)
embeddings.forward(t)

tensor([-2.6237e-01, -9.9466e-01, -9.5892e-01, -1.0342e-02,  4.7943e-01,
         9.8753e-01,  4.9979e-02,  9.9988e-01,  5.0000e-03,  1.0000e+00,
         5.0000e-04,  1.0000e+00,  5.0000e-05,  1.0000e+00,  5.0000e-06,
         1.0000e+00])

In [21]:
embeddings = TimeEmbedding(dim=d)
t_test = torch.tensor([0, 50, 500, 750, 999])  # batch of 5 timesteps
out = embeddings(t_test)
print(out.shape)

torch.Size([5, 16])


In [23]:
out

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [-0.2624, -0.1032, -0.9589,  0.9999,  0.4794,  0.1575,  0.0500,  0.0158,
          0.9650, -0.9947,  0.2837, -0.0103,  0.8776,  0.9875,  0.9988,  0.9999],
        [-0.4678,  0.8595, -0.2624, -0.1032, -0.9589,  0.9999,  0.4794,  0.1575,
         -0.8838,  0.5112,  0.9650, -0.9947,  0.2837, -0.0103,  0.8776,  0.9875],
        [ 0.7451, -0.9998, -0.3878, -0.9880,  0.9380,  0.6961,  0.6816,  0.2350,
         -0.6670, -0.0194,  0.9218,  0.1545,  0.3466, -0.7180,  0.7317,  0.9720],
        [-0.0265,  0.9836, -0.5899,  0.1743, -0.5356, -0.0175,  0.8409,  0.3107,
          0.9996, -0.1805,  0.8075,  0.9847, -0.8445, -0.9998,  0.5411,  0.9505]])